In [1]:
import pandas as pd

In [2]:
hot_grouped_data = pd.read_csv(r"C:\Users\HP\Desktop\Projects - Data Analysis\RSF\RSF\hot_encoded_ckm_grouped_by_encounter_date.csv")

hot_grouped_data.columns

Index(['patient_id', 'year_of_birth', 'death_date', 'encounter_id',
       'encounter_datetime', 'discharge_datetime', 'height_value',
       'weight_value', 'bmi_value', 'waist_circumference_value', 'systolic_bp',
       'diastolic_bp', 'heart_rate', 'ALT', 'Fasting Glucose',
       'HDL Cholesterol', 'HbA1c', 'Hemoglobin', 'LDL Cholesterol',
       'Potassium', 'Serum Albumin', 'Serum Creatinine', 'Sodium',
       'Triglycerides', 'UACR', 'Uric Acid', 'eGFR',
       'Acute myocardial infarction, unspecified',
       'Atherosclerotic heart disease, subclinical',
       'Cerebral infarction, unspecified', 'Chronic kidney disease, stage 3',
       'Chronic kidney disease, stage 4', 'Chronic kidney disease, stage 5',
       'Essential (primary) hypertension', 'Heart failure, unspecified',
       'Hyperlipidemia, unspecified', 'Obesity, unspecified',
       'Peripheral artery disease, unspecified extremity', 'Prediabetes',
       'Proteinuria, severely increased',
       'Type 2 diabetes 

In [3]:
def ckm_staging(row):
    egfr= row['eGFR']
    uacr = row['UACR']
    hba1c = row['HbA1c']
    ldl = row['LDL Cholesterol']
    hyperlipidemia = row['Hyperlipidemia, unspecified']
    prediabetes = row['Prediabetes']
    t2dm = row['Type 2 diabetes mellitus without complications']
    male = row['sex_M']
    bmi = row['bmi_value']
    waist = row['waist_circumference_value']
    hf = row['Heart failure, unspecified']
    htn = row['Essential (primary) hypertension']
    atherosclerotic = row['Atherosclerotic heart disease, subclinical']
    stroke = row['Cerebral infarction, unspecified']
    pad = row['Peripheral artery disease, unspecified extremity']

    kdigo_g5 =  egfr < 15
    kdigo_g4 = 15 <= egfr <= 29
    kdigo_g3b = 30 <= egfr <= 44
    kdigo_g3a = 45 <= egfr <= 59
    kdigo_g2 = 60 <= egfr <= 89
    kdigo_g1 = egfr >= 90

    male_waist = False
    female_waist = False

    if  male == 1 and waist >= 102:
        male_waist = True
    if not male and waist >= 88:
        female_waist = True
 
    if hf or stroke or pad:
        return 4
    elif (((kdigo_g5 or kdigo_g4) or (kdigo_g3b and 30 <= uacr <= 299) or 
        (kdigo_g3a and uacr >= 300)) or uacr >= 300) and (
            htn or hba1c > 5.7 or 
            prediabetes or hyperlipidemia or 
            ldl > 2.6) or atherosclerotic:
        return 3
    elif (
        ((kdigo_g3b and uacr < 30) or 
        (kdigo_g3a and uacr <= 299) or
        ((kdigo_g2 or kdigo_g1) and uacr >= 30)) and
        (htn or hba1c > 5.7 or 
         hyperlipidemia or ldl > 2.6 or 
         prediabetes or t2dm)
        ):
        return 2
    elif (
        (bmi >= 25 or male_waist or female_waist or hba1c > 5.7 or prediabetes)
        and (not htn or not hyperlipidemia or not t2dm or not ldl)
        ):
        return 1
    else:
        return 0

hot_ckm_stage = hot_grouped_data.apply(ckm_staging, axis=1)

hot_staged_data = pd.concat([hot_grouped_data, pd.Series(hot_ckm_stage, name= 'ckm_stage')], axis=1)
hot_staged_data


,patient_id,year_of_birth,death_date,encounter_id,encounter_datetime,discharge_datetime,height_value,weight_value,bmi_value,waist_circumference_value,...,sex_M,nationality_Egypt,nationality_India,nationality_Jordan,nationality_Other,nationality_Pakistan,nationality_Philippines,nationality_UAE,nationality_United Kingdom,ckm_stage
0,PT000001,1970,0,ENC00000001,2015,2015,169.7,93.6,32.5,90.2,...,1,0,0,0,0,1,0,0,0,1
1,PT000002,1970,0,ENC00000036,2015,2015,NaN,NaN,NaN,NaN,...,1,0,0,0,0,0,0,0,0,0
2,PT000003,1970,0,ENC00000079,2015,2015,189.7,73.1,20.3,85.5,...,1,0,0,0,0,0,1,0,0,0
3,PT000004,1970,0,ENC00000115,2015,2015,165.0,83.9,30.8,86.8,...,0,0,1,0,0,0,0,0,0,1
4,PT000005,1970,0,ENC00000158,2015,2015,161.1,101.8,39.2,84.2,...,0,0,0,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,PT001196,1970,1,ENC00028566,2015,2015,171.2,52.8,18.0,76.3,...,1,0,1,0,0,0,0,0,0,0
1196,PT001197,1970,0,ENC00028584,2015,2015,172.3,87.4,29.4,73.5,...,1,0,0,0,0,1,0,0,0,1
1197,PT001198,1970,0,ENC00028610,2015,2015,152.0,57.1,24.7,77.7,...,0,0,0,0,0,0,1,0,0,0
1198,PT001199,1970,1,ENC00028624,2015,2015,164.4,81.5,30.2,108.2,...,0,0,0,0,0,0,1,0,0,1


In [4]:
hot_staged_data['ckm_stage'].value_counts()

ckm_stage
1    728
0    453
2     10
3      9
Name: count, dtype: int64

In [5]:
hot_staged_data.to_csv('ckm_staged_ehr_data.csv', index=False)